# 第 2 周练习 —— 技术导师（Gradio 原型）

## 练习目标（理念）

把第 1 周的技术问答工具升级成可点击的 **Gradio 聊天原型**：

- **流式（streaming）**：一边生成一边刷新回复，而不是等整段答完
- **System Prompt**：固定「技术导师」人设与回答风格
- **多模型下拉**：在 `gpt-4o-mini`、`claude-haiku-4.5`、本地 `llama3.2` 之间切换

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Gradio `ChatInterface` | `type="messages"` + `additional_inputs` 下拉 |
| 多提供商 API | OpenAI SDK / Anthropic SDK / Ollama 兼容端点 |
| 流式输出 | `yield` 累积字符串，驱动 UI 实时刷新 |

## 怎么跑

1. 准备 `.env`（至少 OpenAI；可选 Anthropic）；本地模型需 Ollama 在 `localhost:11434`
2. 从上到下运行单元格；可先跑「快速检查」再 `launch()` UI


In [1]:
# ========== 导入 + 三客户端 + 模型表 + System Prompt ==========

# 导入 gradio：快速搭聊天 Web UI（ChatInterface）
import gradio as gr
# 导入 anthropic：官方 SDK，走 Claude Messages API（含流式）
import anthropic
# 从 openai 导入 OpenAI：云端 GPT 与本地 Ollama（兼容 /v1）共用同一客户端类
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv

# override=True：.env 覆盖进程里已有同名环境变量（改完密钥要重跑本格）
load_dotenv(override=True)

# 云端 OpenAI 客户端：默认读 OPENAI_API_KEY
frontier = OpenAI()
# Anthropic 官方客户端：默认读 ANTHROPIC_API_KEY
claude = anthropic.Anthropic()
# 本地 Ollama：OpenAI 兼容端点；api_key 对本地通常任意，URL/密钥字符串保持原样
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 下拉标签 -> (提供商名, 实际 model id)；展示名与真实 id 可以不同（如 Claude）
MODELS = {
    "gpt-4o-mini": ("openai", "gpt-4o-mini"),
    "claude-haiku-4.5": ("anthropic", "claude-haiku-4-5-20251001"),
    "llama3.2": ("ollama", "llama3.2"),
}

# System Prompt：发给模型的人设/风格指令；正文保留英文，改译会改变回答行为
SYSTEM = "You are an expert technical tutor. Explain clearly and concisely, with a short example when useful. Use markdown."


In [2]:
# ========== chat：按提供商分流，流式 yield 累积回复 ==========

def chat(message, history, model_label):
    """Stream a reply. `history` is Gradio's list of {role, content} messages."""
    # 用下拉标签查出提供商与真实 model id
    provider, model = MODELS[model_label]
    # Anthropic 走官方 messages.stream；system 单独参数，不放进 messages
    if provider == "anthropic":
        with claude.messages.stream(
            model=model, system=SYSTEM, max_tokens=1000,
            # history 已是 messages 格式；再拼上当前 user 消息
            messages=history + [{"role": "user", "content": message}],
        ) as stream:
            # reply 累积已生成文本；每次 yield 整段，供 Gradio 刷新气泡
            reply = ""
            for text in stream.text_stream:
                reply += text
                yield reply
    else:
        # OpenAI 与 Ollama 共用 chat.completions；按 provider 选客户端
        client = frontier if provider == "openai" else local
        # OpenAI 风格：system 作为第一条 message
        messages = [{"role": "system", "content": SYSTEM}] + history + [{"role": "user", "content": message}]
        # stream=True：返回可迭代 chunk，而不是一次性完整对象
        stream = client.chat.completions.create(model=model, messages=messages, stream=True)
        reply = ""
        for chunk in stream:
            # delta.content 可能是 None（角色/空包）；用 or "" 安全拼接
            reply += chunk.choices[0].delta.content or ""
            yield reply


## 快速检查：每个模型各问一句

不启动 UI，先对 `MODELS` 里每个标签跑一遍流式 `chat`，只打印最终完整回复，确认密钥/本地服务都通。


In [3]:
# ========== 冒烟测试：遍历 MODELS，吞掉中间 yield，打印最终 reply ==========

for label in MODELS:
    reply = ""
    # history=[]：无多轮上下文；for 只为耗尽生成器，变量 reply 停在最后一次 yield
    for reply in chat("In one sentence, what is a Python decorator?", [], label):
        pass
    # 标签 + 最终回复；f-string 与提问英文保持原样
    print(f"[{label}] {reply}\n")


[gpt-4o-mini] A Python decorator is a design pattern that allows you to modify the behavior of a function or method by wrapping it in another function, typically used to add functionality, like logging or access control, without altering the original function's code.



[claude-haiku-4.5] A decorator is a function that modifies or wraps another function or class to change its behavior without permanently altering its source code.



[llama3.2] A Python decorator is a small function that takes another function as an argument and returns a new function that "wraps" the original function, allowing for the addition of additional functionality or behavior to be applied before or after the execution of the original function.

**Example:**
```python
def my_decorator(func):
    def wrapper():
        print("Before executing the function")
        func()
        print("After executing the function")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()  # Output: Before executing the function, Hello!, After executing the function
```
In this example, `my_decorator` is a decorator that prints a message before and after the execution of the `say_hello()` function. The `@my_decorator` syntax before the function definition allows for easy application of the decorator to existing code.



## 启动 Gradio 聊天界面

`ChatInterface` 把 `chat` 接到浏览器；`additional_inputs` 把模型下拉传给函数第三个参数。


In [ ]:
# ========== Gradio UI：ChatInterface + 模型下拉 ==========

demo = gr.ChatInterface(
    # fn=chat：签名 (message, history, model_label)；与 additional_inputs 对齐
    fn=chat,
    # type="messages"：history 为 [{role, content}, ...]，与 OpenAI/Anthropic 消息形近
    type="messages",
    # 页面标题（展示文案；保持原样）
    title="Technical tutor",
    # 额外输入：下拉选项来自 MODELS 的 key；默认 gpt-4o-mini
    additional_inputs=[gr.Dropdown(list(MODELS), value="gpt-4o-mini", label="Model")],
)
# launch()：起本地 Web 服务并弹出/打印访问地址
demo.launch()
